# Matemáticas de la Inteligencia Artificial
## Sesión 13 — El bloque Transformer pieza a pieza

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/13_transformer/laboratorio.ipynb)

**Pregunta de la sesión:** ¿cómo pasamos de una cabeza de self-attention causal a un bloque Transformer decoder-only completo, transparente y verificable?

En este laboratorio no utilizaremos `nn.MultiheadAttention`, `nn.Transformer`, `nn.TransformerEncoderLayer` ni `nn.TransformerDecoderLayer`. Construiremos explícitamente:

\[
\text{posición}
\rightarrow
\text{multi-head causal attention}
\rightarrow
\text{FFN + GELU}
\rightarrow
\text{residuales}
\rightarrow
\text{LayerNorm}
\rightarrow
\text{bloque decoder-only}.
\]

El objetivo es que cada tensor tenga un significado matemático claro y que el bloque final pase tres auditorías:

1. **dimensiones correctas**;
2. **causalidad**: modificar el futuro no puede alterar el pasado;
3. **ruta residual**: el bloque puede aproximar la identidad si sus subcapas producen correcciones pequeñas.


In [ ]:
import math, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)


## 1. Diccionario matemática ↔ código

Trabajaremos con batches de secuencias.

| Objeto | Símbolo | Forma |
|---|---:|---:|
| secuencia de ids | \(I\) | \((B,T)\) |
| embeddings | \(E\) | \((B,T,d_{\rm model})\) |
| representación con posición | \(X\) | \((B,T,d_{\rm model})\) |
| queries/keys/values por cabeza | \(Q,K,V\) | \((B,h,T,d_h)\) |
| scores | \(S\) | \((B,h,T,T)\) |
| pesos de atención | \(A\) | \((B,h,T,T)\) |
| salida multi-head | \(Y\) | \((B,T,d_{\rm model})\) |
| capa oculta FFN | \(H\) | \((B,T,d_{\rm ff})\) |

Usaremos
\[
d_h=\frac{d_{\rm model}}{h}.
\]

En todo el cuaderno, la posición \(i\) puede atender únicamente a \(j\le i\).


## 2. Embeddings de token y posición

El embedding de token responde a **qué símbolo es**. El embedding posicional responde a **dónde está**.

La entrada de la primera capa será

\[
X = E_{\rm token} + E_{\rm pos}.
\]

Comenzaremos con posiciones aprendidas porque hacen la implementación especialmente transparente.


In [ ]:
vocab = ['<pad>', 'la', 'masa', 'curva', 'el', 'espacio', 'tiempo',
         'luz', 'materia', 'geometria', '.']
stoi = {t:i for i,t in enumerate(vocab)}
itos = {i:t for t,i in stoi.items()}

frases = [
    ['la','masa','curva','el','espacio','.'],
    ['la','materia','curva','el','tiempo','.']
]
ids = torch.tensor([[stoi[t] for t in frase] for frase in frases], dtype=torch.long)

B, T = ids.shape
d_model = 16
max_T = 32

tok_emb = nn.Embedding(len(vocab), d_model)
pos_emb = nn.Embedding(max_T, d_model)

# TODO 1:
# 1) construye las posiciones 0,...,T-1;
# 2) calcula E_token y E_pos;
# 3) suma ambas contribuciones en X.
pos = ...
E_token = ...
E_pos = ...
X = ...

print('ids:', ids.shape)
print('X  :', X.shape)
assert X.shape == (B, T, d_model)


### Comprobación conceptual: el mismo token en posiciones distintas

Si el token `la` aparece en dos posiciones diferentes, su embedding léxico es el mismo, pero su representación inicial completa cambia por el término posicional.


In [ ]:
token_la = tok_emb(torch.tensor([stoi['la'], stoi['la']]))
p0 = pos_emb(torch.tensor([0]))
p3 = pos_emb(torch.tensor([3]))

# TODO 2: construye las dos representaciones completas del mismo token.
x0 = ...
x3 = ...

print('¿embedding léxico idéntico?', torch.allclose(token_la[0], token_la[1]))
print('¿representación completa idéntica?', torch.allclose(x0, x3))
assert torch.allclose(token_la[0], token_la[1])
assert not torch.allclose(x0, x3)


## 3. Codificación sinusoidal: una alternativa no aprendida

El Transformer original propuso

\[
PE_{(\mathrm{pos},2i)}=
\sin\left(\frac{\mathrm{pos}}{10000^{2i/d_{\rm model}}}\right),
\]

\[
PE_{(\mathrm{pos},2i+1)}=
\cos\left(\frac{\mathrm{pos}}{10000^{2i/d_{\rm model}}}\right).
\]

Vamos a implementarla y visualizar sus primeras coordenadas.


In [ ]:
def sinusoidal_encoding(T, d_model, device='cpu'):
    # TODO 3: implementa la matriz PE de forma (T, d_model).
    # Pista: usa torch.arange y separa columnas pares/impares.
    PE = ...
    return PE

PE = sinusoidal_encoding(T=24, d_model=16)
print(PE.shape)

plt.figure(figsize=(8,4))
for j in range(6):
    plt.plot(PE[:,j].cpu(), label=f'dim {j}')
plt.xlabel('posición')
plt.ylabel('valor')
plt.title('Primeras coordenadas de la codificación sinusoidal')
plt.legend()
plt.tight_layout()
plt.show()

assert PE.shape == (24,16)


## 4. De una cabeza a varias: multi-head causal self-attention

Para cada cabeza \(r\),

\[
Q_r=XW^Q_r,\qquad K_r=XW^K_r,\qquad V_r=XW^V_r,
\]

y

\[
H_r=
\mathrm{softmax}\left(
\frac{Q_rK_r^T}{\sqrt{d_h}}+M
\right)V_r.
\]

Después concatenamos todas las cabezas y proyectamos:

\[
\mathrm{MHA}(X)=
\mathrm{Concat}(H_1,\ldots,H_h)W^O.
\]

En código, en lugar de crear \(h\) módulos separados, calcularemos primero una proyección de dimensión \(d_{\rm model}\) y reorganizaremos:

\[
(B,T,d_{\rm model})
\to
(B,h,T,d_h).
\]


In [ ]:
class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        B, T, C = x.shape
        # TODO 4:
        # (B,T,C) -> (B,T,h,d_h) -> (B,h,T,d_h)
        return ...

    def _merge_heads(self, x):
        B, h, T, d_h = x.shape
        # TODO 5:
        # (B,h,T,d_h) -> (B,T,h,d_h) -> (B,T,C)
        return ...

    def forward(self, x, return_attention=False):
        B, T, C = x.shape

        # TODO 6: proyecciones y separación en cabezas.
        Q = ...
        K = ...
        V = ...

        # TODO 7: scores escalados de forma (B,h,T,T).
        scores = ...

        # Máscara causal: True en las conexiones prohibidas j>i.
        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=x.device),
            diagonal=1
        )

        # TODO 8: aplica la máscara antes del softmax y calcula A.
        scores = ...
        A = ...
        A = self.dropout(A)

        # TODO 9: agrega V, reúne cabezas y aplica la proyección de salida.
        H = ...
        Y = ...

        if return_attention:
            return Y, A
        return Y


In [ ]:
mha = MultiHeadCausalAttention(d_model=16, n_heads=4)
Y, A = mha(X, return_attention=True)

print('X:', X.shape)
print('Y:', Y.shape)
print('A:', A.shape)

assert Y.shape == X.shape
assert A.shape == (B, 4, T, T)
assert torch.allclose(
    torch.triu(A, diagonal=1),
    torch.zeros_like(A),
    atol=1e-7
)
assert torch.allclose(A.sum(dim=-1), torch.ones_like(A.sum(dim=-1)), atol=1e-6)


### Visualización: varias geometrías de compatibilidad

Cada cabeza aprende sus propias proyecciones. Por ello, aun con la misma entrada, los mapas de atención pueden ser distintos.


In [ ]:
fig = plt.figure(figsize=(7,7))
for h in range(4):
    ax = fig.add_subplot(2,2,h+1)
    ax.imshow(A[0,h].detach().cpu(), vmin=0, vmax=1)
    ax.set_title(f'cabeza {h}')
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels(frases[0], rotation=45, ha='right')
    ax.set_yticklabels(frases[0])
plt.tight_layout()
plt.show()


## 5. Auditoría de causalidad

Una prueba estructural muy potente consiste en crear dos secuencias iguales hasta una posición \(k\), modificar únicamente el futuro y comprobar que las salidas hasta \(k\) no cambian.

Esto debe cumplirse **sin entrenamiento**: es una propiedad de la arquitectura y de la máscara.


In [ ]:
def causalidad_test(modulo, x, k):
    x2 = x.clone()
    # perturbamos solamente el futuro
    x2[:, k+1:, :] = torch.randn_like(x2[:, k+1:, :]) * 10.0

    with torch.no_grad():
        y1 = modulo(x)
        y2 = modulo(x2)

    error_pasado = (y1[:, :k+1] - y2[:, :k+1]).abs().max().item()
    error_futuro = (y1[:, k+1:] - y2[:, k+1:]).abs().max().item()
    return error_pasado, error_futuro

# TODO 10: ejecuta la prueba para k=2.
error_pasado, error_futuro = ...

print('máxima diferencia en pasado:', error_pasado)
print('máxima diferencia en futuro:', error_futuro)
assert error_pasado < 1e-6


## 6. Feed-forward network posición a posición

Después de que la atención comunique información entre posiciones, cada token procesa su representación de forma no lineal:

\[
\mathrm{FFN}(x)=
W_2\,\mathrm{GELU}(W_1x+b_1)+b_2.
\]

Usaremos la arquitectura expansión–contracción

\[
d_{\rm model}
\to
d_{\rm ff}
\to
d_{\rm model}.
\]


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        # TODO 11: completa la MLP con Linear -> GELU -> Linear -> Dropout.
        self.net = ...

    def forward(self, x):
        return self.net(x)

ffn = FeedForward(d_model=16, d_ff=64)
Z = ffn(X)
print(Z.shape)
assert Z.shape == X.shape


### GELU frente a ReLU

GELU se define por

\[
\mathrm{GELU}(x)=x\Phi(x),
\]

donde \(\Phi\) es la función de distribución acumulada de una normal estándar.

Compararemos numéricamente ambas activaciones.


In [ ]:
xs = torch.linspace(-4,4,400)
relu = F.relu(xs)
gelu = F.gelu(xs)

plt.figure(figsize=(7,4))
plt.plot(xs, relu, label='ReLU')
plt.plot(xs, gelu, label='GELU')
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.legend()
plt.xlabel('x')
plt.ylabel('activación')
plt.title('ReLU y GELU')
plt.tight_layout()
plt.show()

for valor in [-1.0, 0.0, 1.0]:
    t = torch.tensor(valor)
    print(valor, '-> GELU =', F.gelu(t).item())


## 7. LayerNorm desde la fórmula

Para cada token \(x\in\mathbb R^{d_{\rm model}}\),

\[
\mu=\frac1d\sum_j x_j,
\qquad
\sigma^2=\frac1d\sum_j(x_j-\mu)^2,
\]

\[
\widehat x_j=
\frac{x_j-\mu}{\sqrt{\sigma^2+\varepsilon}},
\]

\[
\mathrm{LN}(x)=\gamma\odot \widehat x+\beta.
\]

Primero implementaremos la estandarización manual y la compararemos con `nn.LayerNorm` inicializada con \(\gamma=1\), \(\beta=0\).


In [ ]:
def layer_norm_manual(x, eps=1e-5):
    # TODO 12: media y varianza sobre la última dimensión, conservando ejes.
    mu = ...
    var = ...
    return ...

x_demo = torch.randn(3,5,16)
ln = nn.LayerNorm(16, eps=1e-5)

manual = layer_norm_manual(x_demo)
pytorch = ln(x_demo)

print('máximo error:', (manual-pytorch).abs().max().item())
assert torch.allclose(manual, pytorch, atol=2e-6)


## 8. Conexión residual

Una subcapa residual implementa

\[
y=x+F(x).
\]

Su Jacobiano contiene un término identidad:

\[
\frac{\partial y}{\partial x}=I+J_F.
\]

Vamos a comprobar experimentalmente la idea más sencilla: si anulamos los parámetros de \(F\), la salida residual debe coincidir exactamente con la entrada.


In [ ]:
linear = nn.Linear(16,16, bias=False)
nn.init.zeros_(linear.weight)

x_res = torch.randn(2,4,16)
# TODO 13: construye la salida residual.
y_res = ...

print('error respecto a identidad:', (y_res-x_res).abs().max().item())
assert torch.allclose(y_res, x_res)


## 9. Bloque Transformer decoder-only en versión pre-norm

Usaremos

\[
Y=X+\mathrm{MHA}_{\rm causal}(\mathrm{LN}_1(X)),
\]

\[
Z=Y+\mathrm{FFN}(\mathrm{LN}_2(Y)).
\]

Esta disposición hace muy visible la ruta residual de identidad.


In [ ]:
class TransformerBlockPreNorm(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, return_attention=False):
        # TODO 14: implementa las dos subcapas residuales pre-norm.
        if return_attention:
            a_out, A = ...
            y = ...
            z = ...
            return z, A
        else:
            y = ...
            z = ...
            return z

bloque = TransformerBlockPreNorm(16, 4, 64)
Z, A_bloque = bloque(X, return_attention=True)

print(Z.shape, A_bloque.shape)
assert Z.shape == X.shape
assert A_bloque.shape == (B,4,T,T)


## 10. Comparación conceptual con post-norm

El Transformer original usó

\[
Y=\mathrm{LN}(X+\mathrm{MHA}(X)),
\]

\[
Z=\mathrm{LN}(Y+\mathrm{FFN}(Y)).
\]

Implementaremos también esta variante para poder comparar ambas estructuras.


In [ ]:
class TransformerBlockPostNorm(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.attn = MultiHeadCausalAttention(d_model, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # TODO 15: implementa post-norm.
        y = ...
        z = ...
        return z

bloque_post = TransformerBlockPostNorm(16,4,64)
Z_post = bloque_post(X)
assert Z_post.shape == X.shape


## 11. Apilar bloques

Cada bloque conserva la forma

\[
(B,T,d_{\rm model})
\to
(B,T,d_{\rm model}),
\]

por lo que podemos componerlos.

Construiremos una pequeña pila de \(L\) bloques y una LayerNorm final.


In [ ]:
class MiniTransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.0):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlockPreNorm(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, x):
        # TODO 16: recorre los bloques y aplica ln_f al final.
        for block in self.blocks:
            x = ...
        return ...

backbone = MiniTransformerBackbone(16,4,64,n_layers=3)
H = backbone(X)
print(H.shape)
assert H.shape == X.shape


## 12. Proyección al vocabulario

La representación final de cada posición tiene dimensión \(d_{\rm model}\). Para obtener un logit por token del vocabulario aplicamos

\[
Z = HW_{\rm vocab}+b_{\rm vocab}.
\]

No entrenaremos todavía el modelo completo: eso corresponde a la sesión 14.


In [ ]:
lm_head = nn.Linear(d_model, len(vocab))

# TODO 17: calcula logits y probabilidades.
logits = ...
probs = ...

print('logits:', logits.shape)
print('probs :', probs.shape)
print('suma de probabilidades en [0,0]:', probs[0,0].sum().item())

assert logits.shape == (B,T,len(vocab))
assert torch.allclose(probs.sum(dim=-1), torch.ones(B,T), atol=1e-6)


## 13. Conteo de parámetros

Antes de entrenar conviene saber dónde están los parámetros.

Para una atención multi-head con proyecciones densas \(Q,K,V,O\), ignorando sesgos,

\[
N_{\rm att}\approx 4d_{\rm model}^2.
\]

Para la FFN,

\[
N_{\rm FFN}
=
2d_{\rm model}d_{\rm ff}+d_{\rm ff}+d_{\rm model}.
\]

Compararemos estas fórmulas con PyTorch.


In [ ]:
def nparams(model):
    return sum(p.numel() for p in model.parameters())

att_demo = MultiHeadCausalAttention(16,4)
ff_demo = FeedForward(16,64)

# TODO 18: calcula las fórmulas teóricas.
n_att_teoria = ...
n_ff_teoria = ...

print('Atención: PyTorch =', nparams(att_demo), '| teoría =', n_att_teoria)
print('FFN     : PyTorch =', nparams(ff_demo), '| teoría =', n_ff_teoria)

assert nparams(att_demo) == n_att_teoria
assert nparams(ff_demo) == n_ff_teoria


## 14. Auditoría final automática del bloque

Una implementación correcta debe satisfacer al menos:

- conservar dimensiones;
- producir atención triangular causal;
- normalizar cada fila de atención;
- impedir que una perturbación del futuro modifique las salidas del pasado;
- permitir backpropagation.


In [ ]:
def auditar_bloque(block, B=2, T=7, d_model=16):
    x = torch.randn(B,T,d_model, requires_grad=True)
    y, A = block(x, return_attention=True)

    informe = {}
    informe['shape_ok'] = (y.shape == x.shape)
    informe['causal_weights_ok'] = torch.allclose(
        torch.triu(A, diagonal=1),
        torch.zeros_like(A),
        atol=1e-7
    )
    informe['rows_sum_one'] = torch.allclose(
        A.sum(dim=-1),
        torch.ones_like(A.sum(dim=-1)),
        atol=1e-6
    )

    # TODO 19: prueba de no fuga de futuro para k=3.
    k = 3
    x2 = x.detach().clone()
    x2[:,k+1:,:] = torch.randn_like(x2[:,k+1:,:]) * 50
    with torch.no_grad():
        y1 = block(x.detach())
        y2 = block(x2)
    informe['no_future_leak'] = ...

    # TODO 20: construye una pérdida escalar y ejecuta backward.
    loss = ...
    loss.backward()
    informe['gradients_exist'] = x.grad is not None and torch.isfinite(x.grad).all().item()

    return informe

informe = auditar_bloque(TransformerBlockPreNorm(16,4,64))
print(informe)
assert all(informe.values())


# Problema final abierto — Laboratorio de arquitectura: ¿qué hace estable y causal a un Transformer?

Construye un **experimento de auditoría arquitectónica** que compare una pila pre-norm y una pila post-norm sin entrenarlas todavía.

Debes decidir y justificar:

- \(d_{\rm model}\);
- número de cabezas \(h\);
- \(d_{\rm ff}\);
- profundidad \(L\in\{1,2,4,8\}\).

## Parte A — Causalidad

Para cada profundidad:

1. genera una entrada aleatoria;
2. modifica de forma drástica solamente los tokens posteriores a una posición \(k\);
3. mide la máxima variación en las salidas \(0,\ldots,k\).

Una arquitectura decoder-only correcta debe dar un error numérico compatible con cero.

## Parte B — Propagación del gradiente

Para cada profundidad \(L\):

1. crea una pila nueva con inicialización aleatoria;
2. define una pérdida escalar usando únicamente la última posición, por ejemplo
   \[
   \mathcal L=\|h_T\|^2;
   \]
3. calcula
   \[
   \|\nabla_X\mathcal L\|_2;
   \]
4. repite el experimento varias veces con distintas semillas;
5. compara pre-norm y post-norm.

No buscamos demostrar un teorema a partir de pocas simulaciones. Buscamos **formular una hipótesis**, apoyarla o refutarla con evidencia y explicar qué rasgo de la arquitectura puede producir la diferencia.

## Parte C — Multi-head

Selecciona una ejecución y representa los mapas de atención de todas las cabezas de una capa. Discute:

- si todas las cabezas producen el mismo patrón;
- por qué no debemos asignar automáticamente una interpretación lingüística a una cabeza no entrenada;
- qué información matemática sí podemos leer legítimamente de \(A_{r,i,j}\).

## Entrega

Elabora una conclusión breve que responda:

> **¿Qué propiedades del bloque Transformer son consecuencias exactas de su arquitectura y cuáles son comportamientos empíricos que solo podemos estudiar mediante experimentos o entrenamiento?**

Incluye código, resultados, gráficas y razonamiento. No hay una única elección correcta de hiperparámetros: la calidad de la justificación forma parte del problema.


## Checklist antes de terminar

Debes poder explicar sin mirar el código:

\[
X
\overset{\mathrm{LN}}{\longrightarrow}
Q,K,V
\overset{\mathrm{MHA\ causal}}{\longrightarrow}
X+\Delta X
\overset{\mathrm{LN}}{\longrightarrow}
\mathrm{FFN}
\longrightarrow
X+\Delta X+\Delta X_{\rm FFN}.
\]

Y debes poder responder:

1. ¿qué dimensión se divide entre cabezas?;
2. ¿en qué eje actúa LayerNorm?;
3. ¿por qué la máscara se aplica antes de softmax?;
4. ¿por qué MHA y FFN regresan a \(d_{\rm model}\)?;
5. ¿qué diferencia estructural hay entre pre-norm y post-norm?;
6. ¿qué propiedad de causalidad puede comprobarse sin entrenar?
